In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import os
from dataclasses import dataclass
from tabulate import tabulate
import spacy
import string
from nltk.stem import RSLPStemmer
import nltk
nltk.download('rslp')

[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


True

In [2]:
@dataclass
class Comentario:
    autor: str      # Nome do autor do comentário
    nota: float     # Nota atribuída pelo espectador
    conteudo: str   # Texto do comentário

In [3]:
def extrair_comentarios(url_filme, max_comentarios=40):

    comentarios = []

    # URL para a seção de críticas de espectadores deste filme
    url = url_filme + "/criticas/espectadores/"

    # Realiza uma requisição HTTP para a página de críticas
    response = requests.get(url)
    if response.status_code != 200:
        print(f"Erro ao acessar a página: {url}")
        return comentarios

    content = response.content

    # Cria um objeto BeautifulSoup para analisar o HTML da página
    soup = BeautifulSoup(content, 'html.parser')

    # Busca o elemento de paginação para determinar quantas páginas de comentários existem
    pagination_div = soup.find('div', class_='pagination-item-holder')

    # Se não encontrar paginação, assume que há apenas uma página
    if pagination_div is None:
        last_page = 1
    else:
        pagination_items = pagination_div.find_all(class_='item')
        if len(pagination_items) == 0:
            last_page = 1
        else:
            # Pega o número da última página de comentários
            try:
                last_page = int(pagination_items[-1].text)
            except (ValueError, IndexError):
                last_page = 1

    # Variável para controlar o número de comentários extraídos
    comentarios_extraidos = 0

    # Percorre as páginas de comentários até atingir o limite
    for i in range(1, last_page + 1):
        if comentarios_extraidos >= max_comentarios:
            break

        # Monta a URL para a página atual
        url_pagina = url + f"?page={i}"
        response = requests.get(url_pagina)
        content = response.content
        soup = BeautifulSoup(content, 'html.parser')

        # Encontra todos os cards de comentários na página
        cards = soup.find_all(class_='review-card')

        # Processa cada card de comentário individualmente
        for card in cards:
            if comentarios_extraidos >= max_comentarios:
                break

            try:
                titulo = card.find('div', class_='meta-title')
                nome = titulo.find('span').text.strip()

                # Extrai a nota atribuída pelo usuário ao filme
                nota_elem = card.find(class_='stareval-note')
                if nota_elem:
                    nota = nota_elem.text.strip().replace(',', '.')
                    nota = float(nota)
                else:
                    nota = 0.0

                # Extrai o conteúdo textual da crítica
                conteudo_elem = card.find(class_='review-card-content')
                if conteudo_elem:
                    conteudo = conteudo_elem.text.strip()
                else:
                    conteudo = ""

                # Cria um objeto Comentario e adiciona à lista
                comentario = Comentario(autor=nome, nota=nota, conteudo=conteudo)
                comentarios.append(comentario)
                comentarios_extraidos += 1

            except Exception as e:
                print(f"Erro ao processar comentário: {e}")
                continue
    return comentarios

urls = []
URL = "https://www.adorocinema.com/filmes/melhores/adorocinema/?page="

for i in range(1, 7):
    response = requests.get(URL + str(i))
    content = response.content

    soup = BeautifulSoup(content, 'html.parser')
    links = soup.find_all('a', class_='meta-title-link')

    for link in links:
        url = 'https://www.adorocinema.com' + link.get('href')
        response = requests.get(url + 'criticas-adorocinema/')
        soup = BeautifulSoup(content, 'html.parser')
        urls.append(url)


comentarios = []
for filme_url in urls:
    if filme_url:
        comentarios.append(extrair_comentarios(filme_url))


print("Extração de comentários concluída!")

KeyboardInterrupt: 

In [ ]:
def salvar_dados_processados_csv(comentarios_processados,
                                 caminho_arquivo_comentarios_processados='comentarios.csv'):
    diretorio_comentarios = os.path.dirname(caminho_arquivo_comentarios_processados)


    if diretorio_comentarios and not os.path.exists(diretorio_comentarios):
        os.makedirs(diretorio_comentarios)

    # Lista para armazenar dados dos filmes processados
    dados_filmes_processados = []

    # Lista para armazenar dados dos comentários processados
    dados_comentarios_processados = []

    # Para cada comentário processado, adiciona seus dados no CSV de comentários processados
    for comentario in comentarios_processados:
        linha_comentario = {
            'autor_comentario': comentario.autor,
            'nota_comentario': comentario.nota,
            'conteudo_comentario': comentario.conteudo,
        }
        dados_comentarios_processados.append(linha_comentario)
    # Cria DataFrames com os dados processados
    df_filmes_processados = pd.DataFrame(dados_filmes_processados)
    df_comentarios_processados = pd.DataFrame(dados_comentarios_processados)
    # Salva os DataFrames em arquivos CSV
    df_comentarios_processados.to_csv(caminho_arquivo_comentarios_processados, index=False, encoding='utf-8-sig')
    return caminho_arquivo_comentarios_processados

In [ ]:

salvar_dados_processados_csv(comentarios)
